## Imports

In [187]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

## Data Cleaning and Loading

In [188]:
input_url = "D:/Assignments/Semester-5/Machine-Learning/AFL2/data/original_data.csv"
output_url = "D:/Assignments/Semester-5/Machine-Learning/AFL2/data/cleaned_data.csv"

df = pd.read_csv(input_url, delimiter=';', encoding='latin1')

df.to_csv(output_url, index=False)

df.head()

,id,category,title,body,amenities,bathrooms,bedrooms,currency,fee,has_photo,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,"time,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,"
0,5668626895,housing/rent/apartment,"Studio apartment 2nd St NE, Uhland Terrace NE,...","This unit is located at second St NE, Uhland T...",NaN,NaN,0,USD,No,Thumbnail,...,$790,Monthly,101,NaN,Washington,DC,38.9057,-76.9861,RentLingo,"1577359415,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,..."
1,5664597177,housing/rent/apartment,Studio apartment 814 Schutte Road,"This unit is located at 814 Schutte Road, Evan...",NaN,NaN,1,USD,No,Thumbnail,...,$425,Monthly,106,814 Schutte Rd,Evansville,IN,37.9680,-87.6621,RentLingo,"1577017063,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,..."
2,5668626833,housing/rent/apartment,"Studio apartment N Scott St, 14th St N, Arling...","This unit is located at N Scott St, 14th St N,...",NaN,1,0,USD,No,Thumbnail,...,"$1,390",Monthly,107,NaN,Arlington,VA,38.8910,-77.0816,RentLingo,"1577359410,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,..."
3,5659918074,housing/rent/apartment,Studio apartment 1717 12th Ave,"This unit is located at 1717 12th Ave, Seattle...",NaN,1,0,USD,No,Thumbnail,...,$925,Monthly,116,1717 12th Avenue,Seattle,WA,47.6160,-122.3275,RentLingo,"1576667743,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,..."
4,5668626759,housing/rent/apartment,"Studio apartment Washington Blvd, N Cleveland ...","This unit is located at Washington Blvd, N Cle...",NaN,NaN,0,USD,No,Thumbnail,...,$880,Monthly,125,NaN,Arlington,VA,38.8738,-77.1055,RentLingo,"1577359401,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,..."


In [189]:
df.describe()

,price,longitude
count,10000.000000,9944.000000
mean,1482.881600,-94.675157
std,1074.952131,15.762033
min,200.000000,-158.022100
25%,945.000000,-101.301700
50%,1267.500000,-93.651600
75%,1695.000000,-82.313000
max,52500.000000,-70.191600


# Data Preprocessing

In [190]:
print(df.isnull().sum())

# Fill columns with many missing values with default values
df["amenities"] = df["amenities"].fillna("None")
df["pets_allowed"] = df["pets_allowed"].fillna("Not specified")

# < 1% missing values, safe to drop rows
rows_to_drop = ["bedrooms", "bathrooms", "cityname", "state", "latitude", "longitude"]
df = df.dropna(subset=rows_to_drop)

columns_to_drop = [
    "id",  # Unique identifier, not useful for analysise
    "title",  # Unstructured text, not useful for analysis
    "body",  # Unstructured text, not useful for analysis
    "price_display",  # Redundant with 'price'
    "address",  # Redundant with 'cityname', 'state', 'latitude', 'longitude'
    "cityname",  # Inferior to 'latitude' and 'longitude' for location analysis
    "state",  # Inferior to 'latitude' and 'longitude' for location analysis
    "source",  # Not relevant to the apartment itself
]

columns_that_exist = [col for col in columns_to_drop if col in df.columns]

if columns_that_exist:
    df = df.drop(columns=columns_that_exist)
    print(f"\nDropped columns: {columns_that_exist}")
else:
    print("\nColumns already dropped.")

id                                                                               0
category                                                                         1
title                                                                            0
body                                                                             0
amenities                                                                     3548
bathrooms                                                                       34
bedrooms                                                                         7
currency                                                                        24
fee                                                                              0
has_photo                                                                        0
pets_allowed                                                                  4139
price                                                                            0
pric

In [191]:
print(f"New DataFrame shape: {df.shape}")
print("\nRemaining columns:")
print(df.info())

New DataFrame shape: (9837, 14)

Remaining columns:
<class 'pandas.core.frame.DataFrame'>
Index: 9837 entries, 2 to 9999
Data columns (total 14 columns):
 #   Column                                                                      Non-Null Count  Dtype  
---  ------                                                                      --------------  -----  
 0   category                                                                    9837 non-null   object 
 1   amenities                                                                   9837 non-null   object 
 2   bathrooms                                                                   9837 non-null   object 
 3   bedrooms                                                                    9837 non-null   object 
 4   currency                                                                    9837 non-null   object 
 5   fee                                                                         9837 non-null   object 
 6   h

In [192]:
# Check for columns with low variance or single unique value
if "category" in df.columns:
    print("", df["category"].value_counts())
else:
    print("Category column already dropped.")

if "currency" in df.columns:
    print("\n", df["currency"].value_counts())
else:
    print("Currency column already dropped.")

if "fee" in df.columns:
    print("\n", df["fee"].value_counts())
else:
    print("\nFee column already dropped.")

if "price_type" in df.columns:
    print("\n", df["price_type"].value_counts())
else:
    print("\nPrice type column already dropped.")

if df["has_photo"].dtype == "object":
    print("\n", df["has_photo"].value_counts())
    df["has_photo"] = df["has_photo"].map({"Yes": 1, "Thumbnail": 1, "No": 0})

    print("\nCleaned values:", df["has_photo"].value_counts().to_dict())
else:
    print(
        "\nCleaned values (already converted):",
        df["has_photo"].value_counts().to_dict(),
    )

 category
housing/rent/apartment     9834
housing/rent/home             2
housing/rent/short_term       1
Name: count, dtype: int64

 currency
USD    9837
Name: count, dtype: int64

 fee
No    9837
Name: count, dtype: int64

 price_type
Monthly    9836
Weekly        1
Name: count, dtype: int64

 has_photo
Thumbnail    8768
Yes           887
No            182
Name: count, dtype: int64

Cleaned values: {1: 9655, 0: 182}


In [193]:
# Drop low variance / single value columns
if "category" in df.columns and df["category"].value_counts().iloc[0] / len(df) > 0.99:
    df = df.drop(columns=["category"])
    print("\nDropped 'category' column (99%+ same value).")

if "currency" in df.columns and df["currency"].nunique() == 1:
    df = df.drop(columns=["currency"])
    print("\nDropped 'currency' column (only 1 value).")

if "fee" in df.columns and df["fee"].nunique() == 1:
    df = df.drop(columns=["fee"])
    print("\nDropped 'fee' column (only 1 value).")

if "price_type" in df.columns and (
    df["price_type"].value_counts().iloc[0] / len(df) > 0.99
):
    df = df.drop(columns=["price_type"])
    print("\nDropped 'price_type' column (99%+ same value).")


Dropped 'category' column (99%+ same value).

Dropped 'currency' column (only 1 value).

Dropped 'fee' column (only 1 value).

Dropped 'price_type' column (99%+ same value).


In [194]:
print(f"New DataFrame shape: {df.shape}")
print("\nRemaining columns:")
print(df.info())

New DataFrame shape: (9837, 10)

Remaining columns:
<class 'pandas.core.frame.DataFrame'>
Index: 9837 entries, 2 to 9999
Data columns (total 10 columns):
 #   Column                                                                      Non-Null Count  Dtype  
---  ------                                                                      --------------  -----  
 0   amenities                                                                   9837 non-null   object 
 1   bathrooms                                                                   9837 non-null   object 
 2   bedrooms                                                                    9837 non-null   object 
 3   has_photo                                                                   9837 non-null   int64  
 4   pets_allowed                                                                9837 non-null   object 
 5   price                                                                       9837 non-null   int64  
 6   s

In [195]:
# Convert numeric columns stored as objects to proper numeric types
cols_to_convert = ["bathrooms", "bedrooms", "square_feet", "latitude"]

final_cols_to_convert = []
for col_name in cols_to_convert:
    if col_name in df.columns and df[col_name].dtype == "object":
        final_cols_to_convert.append(col_name)

if final_cols_to_convert:
    print(f"Converting columns to numeric: {final_cols_to_convert}")
    # Force all columns to numeric, setting errors='coerce' will turn bad data into NaN
    df[final_cols_to_convert] = df[final_cols_to_convert].apply(
        pd.to_numeric, errors="coerce"
    )

    # Remove rows with NaN in these columns
    original_rows = len(df)
    df = df.dropna(subset=final_cols_to_convert)
    new_rows = len(df)
    print(
        f"Dropped {original_rows - new_rows} rows due to bad data (e.g., text in a number column)."
    )

    try:
        df["bedrooms"] = df["bedrooms"].astype(int)
        df["square_feet"] = df["square_feet"].astype(int)
        print("Set 'bedrooms' and 'square_feet' to integer type.")
    except Exception as e:
        print(f"Could not set all integer types: {e}")

else:
    print("Numeric columns are already in the correct format.")

Converting columns to numeric: ['bathrooms', 'bedrooms', 'square_feet', 'latitude']
Dropped 0 rows due to bad data (e.g., text in a number column).
Set 'bedrooms' and 'square_feet' to integer type.


In [196]:
print(f"New DataFrame shape: {df.shape}")
print("\nRemaining columns:")
print(df.info())

New DataFrame shape: (9837, 10)

Remaining columns:
<class 'pandas.core.frame.DataFrame'>
Index: 9837 entries, 2 to 9999
Data columns (total 10 columns):
 #   Column                                                                      Non-Null Count  Dtype  
---  ------                                                                      --------------  -----  
 0   amenities                                                                   9837 non-null   object 
 1   bathrooms                                                                   9837 non-null   float64
 2   bedrooms                                                                    9837 non-null   int64  
 3   has_photo                                                                   9837 non-null   int64  
 4   pets_allowed                                                                9837 non-null   object 
 5   price                                                                       9837 non-null   int64  
 6   s

## Data Transformation and Feature Engineering

In [197]:
# Transform 'pets_allowed' column into binary columns
if "pets_allowed" in df.columns:
    print(df["pets_allowed"].value_counts())

    df["pets_allowed"] = df["pets_allowed"].str.replace('"', "")

    df["pets_cats"] = df["pets_allowed"].str.contains("Cats", case=False).astype(int)
    df["pets_dogs"] = df["pets_allowed"].str.contains("Dogs", case=False).astype(int)

    df["pets_none"] = (
        df["pets_allowed"].str.contains("Not specified", case=False).astype(int)
    )

    df = df.drop(columns=["pets_allowed"])

    print("\nCreated 'pets_cats', 'pets_dogs', and 'pets_none' columns.")

    print(f"\nTotal apartments allowing cats: {df['pets_cats'].sum()}")
    print(f"Total apartments allowing dogs: {df['pets_dogs'].sum()}")
    print(
        f"Total apartments allowing no pets (or not specified): {df['pets_none'].sum()}"
    )

    both_pets = df[(df["pets_cats"] == 1) & (df["pets_dogs"] == 1)]
    print(f"Total apartments allowing BOTH cats and dogs: {len(both_pets)}")
else:
    print("'pets_allowed' column already transformed.")

pets_allowed
Cats,Dogs        5147
Not specified    4084
Cats              481
Dogs              123
Cats",Dogs          2
Name: count, dtype: int64

Created 'pets_cats', 'pets_dogs', and 'pets_none' columns.

Total apartments allowing cats: 5630
Total apartments allowing dogs: 5272
Total apartments allowing no pets (or not specified): 4084
Total apartments allowing BOTH cats and dogs: 5149


In [198]:
# Transform 'amenities' column into binary columns
if "amenities" in df.columns:
    # Count using a function to use with apply
    def count_amenities(amenity_string):
        if amenity_string == "None":
            return 0
        else:
            return len(amenity_string.split(","))

    df["amenities_count"] = df["amenities"].apply(count_amenities)

    df = df.drop(columns=["amenities"])

    print("Created 'amenities_count' column.")

    print("\n", df["amenities_count"].value_counts().head())
else:
    print("'amenities' column already transformed.")

Created 'amenities_count' column.

 amenities_count
0    3465
3    1045
2     908
4     820
1     785
Name: count, dtype: int64


In [199]:
print(f"New DataFrame shape: {df.shape}")
print("\nRemaining columns:")
print(df.info())

New DataFrame shape: (9837, 12)

Remaining columns:
<class 'pandas.core.frame.DataFrame'>
Index: 9837 entries, 2 to 9999
Data columns (total 12 columns):
 #   Column                                                                      Non-Null Count  Dtype  
---  ------                                                                      --------------  -----  
 0   bathrooms                                                                   9837 non-null   float64
 1   bedrooms                                                                    9837 non-null   int64  
 2   has_photo                                                                   9837 non-null   int64  
 3   price                                                                       9837 non-null   int64  
 4   square_feet                                                                 9837 non-null   int64  
 5   latitude                                                                    9837 non-null   float64
 6   l